### Preparing Jsonl 

In [16]:
import pandas as pd
import json

In [17]:
import os
os.getcwd()

'<PROJECT_DIR>'

In [18]:
df = pd.read_csv("../scraped_data/scraped_data.csv")  
df = df.dropna(subset=["text","title"]) 
#df = df.head(22494) #first batch of 3
#df= df.iloc[22494:22494*2] #second batch of 3
#df = df.iloc[22494*2:67484]  #third batch of 3 

In [19]:
# === System Prompt ===
system_prompt = (
    "You are a political analyst analyzing news articles about Taiwan from U.S. media. "
    "**Important:** If the article is not substantively about Taiwan, respond with only this JSON: "
    "{\"skip\": true, \"reason\": \"Taiwan not substantively discussed.\"}. "
    "Otherwise, analyze the article carefully and respond to all questions. "
    "Use a 1–5 scale for opinion and stance questions, and 0–100% for topic relevance questions. "
    "Return only a flat JSON object with no markdown, no code fences, and no explanations."
)

# === User Prompt Template ===
user_prompt_template = '''
Article title: {title}

Here is the article content:

{article_text}

Please answer the following questions:

Identification
1. Author name (if unknown, return null)
2. List up to 3 most cited actors driving the events/issues (name + role)
3. List up to 3 most cited sources of information (name + role)

Conflict
4. How strongly does the article convey a sense of conflict intensity and crisis urgency? (1–5)
5. Is China described as provoking or initiating the conflict? (1–5)
6. Is Taiwan described as provoking or initiating the conflict? (1–5)
7. Is another actor cast as the source of the conflict? (1–5)

Otherness
8. How much agency does Taiwan have? (1–5)
9. How human is Taiwan? (1–5)
10. How rational is Taiwan? (1–5)

Attitude
11. How likely is the author to support political relations with Taiwan? (1–5)
12. How likely is the author to support economic relations with Taiwan? (1–5)

Topic Classification (0–100%)
13. Taiwan’s foreign policy and international relations
14. Cross-strait relations
15. Taiwan’s domestic politics
16. Taiwan’s response to Covid-19 or other public health issues
17. Taiwan’s semiconductor and technology industry
18. Taiwan’s business, economy or trade
19. Taiwan’s pop culture (music, movies, food, etc.)
20. Taiwan’s traditional culture
21. Environmental or natural disasters in Taiwan

Other Nuances
22. To what extent does the article emphasize Taiwan’s democratic values? (1–5)
23. To what extent is China-Taiwan framed in an aggressor-victim relationship? (1–5)
24. To what extent is Taiwan framed in contrast with China? (1–5)
25. How much prior knowledge does the article assume the reader already has about Taiwan? (1–5)

Return only a flat valid JSON object with the following keys:
{{
  "author": string or null,
  "top_actor_1": string,
  "top_actor_2": string,
  "top_actor_3": string,
  "top_source_1": string,
  "top_source_2": string,
  "top_source_3": string,
  "conflict_intensity": int,
  "china_as_provocateur": int,
  "taiwan_as_provocateur": int,
  "other_actor_as_provocateur": int,
  "taiwan_agency": int,
  "taiwan_humanity": int,
  "taiwan_rationality": int,
  "support_political_taiwan": int,
  "support_economic_taiwan": int,
  "topic_foreign_policy": int,
  "topic_cross_strait": int,
  "topic_domestic_politics": int,
  "topic_health": int,
  "topic_semiconductors": int,
  "topic_business": int,
  "topic_pop_culture": int,
  "topic_traditional_culture": int,
  "topic_environment": int,
  "emphasis_democracy": int,
  "aggressor_victim_frame": int,
  "taiwan_china_contrast": int,
  "assumed_prior_knowledge": int
}}
'''

In [20]:
# Redo Identification Part 
# === System Prompt ===
system_prompt= (
    "You are a political analyst analyzing news articles about Taiwan from U.S. media. "
    "**Important:** If the article is not substantively about Taiwan, respond with only this JSON: "
    "{\"skip\": true, \"reason\": \"Taiwan not substantively discussed.\"}. "
    "Otherwise, analyze the article carefully and respond to all questions. "
    "Return only a flat JSON object with no markdown, no code fences, and no explanations."
)

# === User Prompt Template ===
user_prompt_template= '''
Article title: {title}

Here is the article content:

{article_text}

Please answer the following questions:
1. List up to 3 most cited actors  driving the events/issue. Only include the name of the person or entity in its most common format without any additional information.
2. List up to 3 most cited sources of information. Only include the name of the person or entity in its most common format without any additional information.

Return only a flat valid JSON object with the following keys:
{{
  "author": string or null,
  "top_actor_1": string,
  "top_actor_2": string,
  "top_actor_3": string,
  "top_source_1": string,
  "top_source_2": string,
  "top_source_3": string, }}
'''

In [21]:
# === Step 3: Create the JSONL batch file ===
output_path = "rebatch_input3.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for i, row in df.iterrows():
        article_id = row.get("sid", f"article-{i}")
        title = row["title"]
        text = row["text"]

        user_prompt = user_prompt_template.format(
            title=title.strip(),
            article_text=text.strip()
        )

        request_payload = {
            "custom_id": str(article_id),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o",  # or "gpt-4-turbo" or "gpt-3.5-turbo"
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                "temperature": 0,
                "max_tokens": 1500
            }
        }

        f.write(json.dumps(request_payload) + "\n")

print(f"✅ Done! Saved {len(df)} requests to {output_path}")


✅ Done! Saved 67483 requests to rebatch_input3.jsonl


### Retriving Results

In [51]:
import json

results = []
file_paths = [
    "Batching_Jsonl_Files/Final/Output_A.jsonl",
    "Batching_Jsonl_Files/Final/Output_B.jsonl",
    "Batching_Jsonl_Files/Final/Output_C.jsonl"
]

for path in file_paths:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            results.append(data)

print(f"✅ Loaded {len(results)} results from {len(file_paths)} files.")


✅ Loaded 67470 results from 3 files.


In [52]:
import json

results = []
file_paths = [
    "Batching_Jsonl_Files/Final/rebatch_output1.jsonl",
    "Batching_Jsonl_Files/Final/rebatch_output2.jsonl",
    "Batching_Jsonl_Files/Final/rebatch_output3.jsonl"
]

for path in file_paths:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            results.append(data)

print(f"✅ Loaded {len(results)} results from {len(file_paths)} files.")


✅ Loaded 67482 results from 3 files.


In [53]:
for i, res in enumerate(results[:50]):
    custom_id = res.get("custom_id")
    response = res.get("response", {})
    body = response.get("body", {})
    choices = body.get("choices", [])
    if choices:
        content = choices[0].get("message", {}).get("content", "")
        print(f"ID: {custom_id} \nAnswer: {content}\n---")


ID: b6588993d863f8607131226513d3945c8e635ebec002a26b1df94fb21a2d9d16 
Answer: {"skip": true, "reason": "Taiwan not substantively discussed."}
---
ID: c9e1d0a8ffd219b20018cc64f744773d0b6c6f99300b27fec9d1cf1575b270de 
Answer: {"skip": true, "reason": "Taiwan not substantively discussed."}
---
ID: d1c10b362cdfabb43d264de7e151b0852440e72cfc9284ab2393e027a4c332a3 
Answer: {"skip": true, "reason": "Taiwan not substantively discussed."}
---
ID: c71b80d7506d468f75f501a08e5fc1b853903aba9fa6cbc7a3afeb8654f58c64 
Answer: {"author":null,"top_actor_1":"China","top_actor_2":"Taiwan","top_actor_3":"People’s Liberation Army","top_source_1":"Taipei-based China Times","top_source_2":"unnamed military officials","top_source_3":""}
---
ID: d04b78a279f1e0764a816dcb06951c328f22b33874f0d1f9435d42002d5512be 
Answer: {"skip": true, "reason": "Taiwan not substantively discussed."}
---
ID: 9c80db10c1f37f1fa17dff00c49e3137babb133d5478112e3c2c57a4387fa4c4 
Answer: {"skip": true, "reason": "Taiwan not substantively

In [ ]:
import json
import pandas as pd

# Step 1: Load all results from multiple files
results = []
file_paths = [
    "Batching_Jsonl_Files/Final/rebatch_output1.jsonl",
    "Batching_Jsonl_Files/Final/rebatch_output2.jsonl",
    "Batching_Jsonl_Files/Final/rebatch_output3.jsonl"
]

for path in file_paths:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            results.append(data)

print(f"✅ Loaded {len(results)} results from {len(file_paths)} files.")

# Step 2: Filter and parse valid responses (not skipped)
valid_data = []
skipped_count = 0
invalid_format_count = 0

for res in results:
    response = res.get("response", {})
    content = response.get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
    try:
        answer_json = json.loads(content)
        if answer_json.get("skip") is not True:
            answer_json["id"] = res.get("custom_id")
            valid_data.append(answer_json)
        else:
            skipped_count += 1
    except json.JSONDecodeError:
        invalid_format_count += 1
        print(f"⚠️ JSON decode error for ID {res.get('custom_id')}")

# Step 3: Create DataFrame
dfbrr = pd.DataFrame(valid_data)

# Step 4: Print summary
print(f"✅ Valid (not skipped): {len(valid_data)}")
print(f"🚫 Skipped: {skipped_count}")
print(f"❌ Invalid JSON: {invalid_format_count}")

# Step 5: (Optional) preview results
dfbrr[["top_actor_1","top_actor_2", "top_actor_3"]].iloc[:20,]

## Network: edge list

In [142]:
import numpy as np
# Which entities the co-mention network is built from:
#   "source" -> top_source_1..3  (quoted/attributed sources; paper Fig 3 / Table B2)
#   "actor"  -> top_actor_1..3   (actors;                    paper Fig 2 / Table B1)
NETWORK = "source"
ppl_cols = [f"top_{NETWORK}_{k}" for k in (1, 2, 3)]   # by name, not by column position
ppl = np.hstack( dfbrr[ppl_cols].values )
ppl_lower = pd.Series([str(x).lower() for x in ppl])
ppl_counts = ppl_lower.value_counts()
print("Most frequent value (dropped by [1:] below):", repr(ppl_counts.index[0]), ppl_counts.iloc[0])
ppl_sub = set(ppl_counts[ppl_counts > 50].index[1:])   # entities cited > 50 times


In [144]:
dfbrr[ppl_cols].iloc[0]

top_source_1      Taipei-based China Times
top_source_2    unnamed military officials
top_source_3                              
Name: 0, dtype: object

In [ ]:
from itertools import combinations

list(combinations([d for d in dfbrr[ppl_cols].iloc[0] if str(d).lower() in ppl_sub],2) )

In [146]:
edge_list = []
for i in range(len(dfbrr)):
    # Lowercase each node before checking membership in ppl_sub
    nodes = [str(d).lower() for d in dfbrr[ppl_cols].iloc[i] if str(d).lower() in ppl_sub]
    edge_list.append(list(combinations(nodes, 2)))

#print(edge_list)

In [147]:
flat_edge_list = [edge for sublist in edge_list for edge in sublist]

In [148]:
from collections import Counter
import netwulf

# Flatten nested edge lists
flat_edge_list = [edge for sublist in edge_list for edge in sublist]

# Count each unique edge (unordered, so (A,B) == (B,A) if undirected)
# If your graph is directed, remove the `tuple(sorted(...))` part.
edge_counts = Counter(tuple(sorted(edge)) for edge in flat_edge_list)

# Extract unique nodes
nodes_set = set()
for source, target in edge_counts:
    nodes_set.update([source, target])

# Create node dicts
nodes = [{"id": node} for node in nodes_set]

# Create weighted edge dicts
links = [{"source": source, "target": target, "weight": weight} 
         for (source, target), weight in edge_counts.items()]

# Prepare network dictionary
network = {"nodes": nodes, "links": links}

# Visualize
netwulf.visualize(network)

(None, None)

## Conslidate 合併相似 sources

In [126]:
unique_nodes = set()
for source, target in flat_edge_list:
    unique_nodes.update([source, target])

num_nodes = len(unique_nodes)
print(f"Number of unique nodes: {num_nodes}")

for nn in sorted(unique_nodes):
    print(nn)

Number of unique nodes: 59

alex azar
antony blinken
apple
australia
beijing
biden administration
chen shih-chung
chinese communist party
chinese government
chinese military
david chou
democratic progressive party
donald trump
federal reserve
foxconn
fumio kishida
han kuo-yu
jake sullivan
japan
joe biden
john cena
joseph wu
kevin mccarthy
kuomintang
lloyd austin
michael mccaul
mike gallagher
mike pompeo
nancy pelosi
pentagon
people's liberation army
people's republic of china
people’s liberation army
philippines
qin gang
russia
taiwan
taiwan government
taiwan semiconductor manufacturing co.
taiwan semiconductor manufacturing company
taiwan stock exchange
taiwan's defense ministry
taiwanese government
trump administration
tsai ing-wen
tsmc
u.s.
u.s. military
u.s. navy
united microelectronics corporation
united states
vladimir putin
wall street
wang yi
white house
william lai
world health organization
xi jinping


In [64]:
#for sources 
consolidation_dict = {
    # Taiwan 
    'taiwanese government': 'taiwan',
    'taiwan government': 'taiwan',

    #variations of Taiwan ministry of national defense 

    'taiwan ministry of national defense': "taiwan's defense ministry",
    'taiwan ministry of national defence': "taiwan's defense ministry",
    'taiwanese defense ministry': "taiwan's defense ministry",
    'taiwan’s ministry of national defense': "taiwan's defense ministry",  # curly
    "taiwan's ministry of national defense": "taiwan's defense ministry",  # straight
    'taiwan defense ministry': "taiwan's defense ministry",
    'taiwan\'s defence ministry':"taiwan's defense ministry",
    'taiwan’s defense ministry' :"taiwan's defense ministry",
    'taiwan’s defense ministry': "taiwan's defense ministry",
    


    # China and variants consolidated to 'china'
    'china\'s foreign ministry': 'chinese foreign ministry',
    'china’s foreign ministry': 'chinese foreign ministry',
    'chinese foreign ministry': 'chinese foreign ministry',

    'chinese communist party': 'china',
    'chinese defense ministry': 'china',
    'chinese embassy': 'china',
    'chinese government': 'china',
    'chinese military': 'china',
    'chinese state media': 'china',
    'beijing':'china',
    
    # Duplicates/variants cleaned up
    'people’s liberation army': 'pla',
    "people's liberation army": 'pla',
    'people\'s liberation army': 'pla',
    'people’s republic of china': 'china',
    "people's republic of china": 'china',
    'people\'s republic of china': 'china',
    
    # Xinhua variations consolidated
    'xinhua': 'xinhua news agency',
    'xinhua news agency': 'xinhua news agency',

    # United States government consolidated to 'us'
    'u.s.': 'us',
    'united states': 'us',
    'u.s. government': 'us',
    'u.s. state department': 'us',

    'u.s. military': 'us military',
    'u.s. navy': 'us military',
    'us navy': 'us military',

    # Other country governments or international organizations
    'australia': 'australia',
    'japan': 'japan',
    'russia': 'russia',
    'philippines': 'philippines',
    'united nations': 'un',
    'world health organization': 'who',
    'nato': 'nato',

    # Media names consolidated for consistency (optional)
    'associated press': 'ap',
    'ap': 'ap',
    'bbc': 'bbc',
    'cbs news': 'cbs news',
    'cctv': 'cctv',
    'cnn': 'cnn',
    'fox news': 'fox news',
    'fox news digital': 'fox news',
    'nbc news': 'nbc news',
    'new york times': 'new york times',
    'the new york times': 'new york times',
    'wall street journal': 'wall street journal',
    'the wall street journal': 'wall street journal',
    'washington post': 'washington post',
    'the washington post': 'washington post',
    'bloomberg': 'bloomberg',
    'reuters': 'reuters',

    # Companies (no consolidation)
    'apple': 'apple',
    'foxconn': 'foxconn',
    'intel': 'intel',
    'taiwan semiconductor manufacturing co': 'tsmc',
    'taiwan semiconductor manufacturing co.': 'tsmc',
    'taiwan semiconductor manufacturing company': 'tsmc',
    'united microelectronics corporation': 'umc',
    'tsmc': 'tsmc',

    # Individual names kept as is to remain independent
    'alex azar': 'alex azar',
    'bonnie glaser': 'bonnie glaser',
    'drew thompson': 'drew thompson',
    'emmanuel macron': 'emmanuel macron',
    'gordon chang': 'gordon chang',
    'han kuo-yu': 'han kuo-yu',
    'jake sullivan': 'jake sullivan',
    'jen psaki': 'jen psaki',
    'joanne ou': 'joanne ou',
    'joe biden': 'joe biden',
    'john cena': 'john cena',
    'john kirby': 'john kirby',
    'karine jean-pierre': 'karine jean-pierre',
    'kevin mccarthy': 'kevin mccarthy',
    'kurt campbell': 'kurt campbell',
    'lev nachman': 'lev nachman',
    'lloyd austin': 'lloyd austin',
    'mark milley': 'mark milley',
    'marsha blackburn': 'marsha blackburn',
    'michael mccaul': 'michael mccaul',
    'mike gallagher': 'mike gallagher',
    'mike pompeo': 'mike pompeo',
    'nancy pelosi': 'nancy pelosi',
    'newt gingrich': 'newt gingrich',
    'nikki haley': 'nikki haley',
    'su tseng-chang': 'su tseng-chang',
    'terry gou': 'terry gou',
    'tom cotton': 'tom cotton',
    'vladimir putin': 'vladimir putin',
    'wang yi': 'wang yi',
    'zhao lijian': 'zhao lijian',
    'xu jinping': 'xi jinping',  # fix possible typo
    'zhu fenglian': 'zhu fenglian',
}

    

    
"""consolidation_dict = {
    # Taiwan and related institutions consolidated to 'taiwan'
    'taiwanese government': 'taiwan',
    'taiwan ministry of foreign affairs': 'taiwan',
    "taiwan's defense ministry": 'taiwan',
    'taiwan ministry of national defense': 'taiwan',
    'taiwan affairs office': 'taiwan',
    'taiwan government': 'taiwan',
    'taipei times': 'taiwan',
    'taiwan relations act': 'us',  # US legislation
    'taiwan ministry of national defence': 'taiwan',
    "taiwan's defence ministry": 'taiwan',
    "taiwan's ministry of national defense": 'taiwan',
    'taiwanese defense ministry': 'taiwan',
    'taiwan’s defense ministry': 'taiwan',
    'taiwan’s ministry of national defense': 'taiwan',

    # China and variants consolidated to 'china'
    'china\'s foreign ministry': 'china',
    'china’s foreign ministry': 'china',
    'chinese communist party': 'china',
    'chinese defense ministry': 'china',
    'chinese embassy': 'china',
    'chinese foreign ministry': 'china',
    'chinese government': 'china',
    'chinese military': 'china',
    'chinese state media': 'china',
    'wang yi': 'china',
    
    # Duplicates/variants cleaned up
    'people’s liberation army': 'pla',
    "people's liberation army": 'pla',
    'people\'s liberation army': 'pla',
    'people’s republic of china': 'china',
    "people's republic of china": 'china',
    'people\'s republic of china': 'china',
    
    # Xinhua variations consolidated
    'xinhua': 'xinhua news agency',
    'xinhua news agency': 'xinhua news agency',

    # United States government consolidated to 'us'
    'antony blinken': 'us',
    'biden administration': 'us',
    'cia': 'us',
    'congress': 'us',
    'department of defense': 'us',
    'pentagon': 'us',
    'white house': 'us',
    'u.s.': 'us',
    'u.s. government': 'us',
    'u.s. military': 'us',
    'u.s. navy': 'us',
    'u.s. state department': 'us',
    'us navy': 'us',
    'united states': 'us',

    # Other country governments or international organizations
    'australia': 'australia',
    'japan': 'japan',
    'russia': 'russia',
    'philippines': 'philippines',
    'united nations': 'un',
    'world health organization': 'who',
    'nato': 'nato',

    # Media names consolidated for consistency (optional)
    'associated press': 'ap',
    'ap': 'ap',
    'bbc': 'bbc',
    'cbs news': 'cbs news',
    'cctv': 'cctv',
    'cnn': 'cnn',
    'fox news': 'fox news',
    'fox news digital': 'fox news',
    'nbc news': 'nbc news',
    'new york times': 'new york times',
    'the new york times': 'new york times',
    'wall street journal': 'wall street journal',
    'the wall street journal': 'wall street journal',
    'washington post': 'washington post',
    'the washington post': 'washington post',
    'bloomberg': 'bloomberg',
    'reuters': 'reuters',

    # Companies (no consolidation)
    'apple': 'apple',
    'foxconn': 'foxconn',
    'intel': 'intel',
    'taiwan semiconductor manufacturing co': 'tsmc',
    'taiwan semiconductor manufacturing co.': 'tsmc',
    'taiwan semiconductor manufacturing company': 'tsmc',
    'united microelectronics corporation': 'umc',
    'tsmc': 'tsmc',

    # Individual names kept as is to remain independent
    'alex azar': 'alex azar',
    'bonnie glaser': 'bonnie glaser',
    'drew thompson': 'drew thompson',
    'emmanuel macron': 'emmanuel macron',
    'gordon chang': 'gordon chang',
    'han kuo-yu': 'han kuo-yu',
    'jake sullivan': 'jake sullivan',
    'jen psaki': 'jen psaki',
    'joanne ou': 'joanne ou',
    'joe biden': 'joe biden',
    'john cena': 'john cena',
    'john kirby': 'john kirby',
    'karine jean-pierre': 'karine jean-pierre',
    'kevin mccarthy': 'kevin mccarthy',
    'kurt campbell': 'kurt campbell',
    'lev nachman': 'lev nachman',
    'lloyd austin': 'lloyd austin',
    'mark milley': 'mark milley',
    'marsha blackburn': 'marsha blackburn',
    'michael mccaul': 'michael mccaul',
    'mike gallagher': 'mike gallagher',
    'mike pompeo': 'mike pompeo',
    'nancy pelosi': 'nancy pelosi',
    'newt gingrich': 'newt gingrich',
    'nikki haley': 'nikki haley',
    'su tseng-chang': 'su tseng-chang',
    'terry gou': 'terry gou',
    'tom cotton': 'tom cotton',
    'vladimir putin': 'vladimir putin',
    'wang yi': 'wang yi',
    'zhao lijian': 'zhao lijian',
    'xu jinping': 'xi jinping',  # fix possible typo
    'zhu fenglian': 'zhu fenglian',
}
"""

'consolidation_dict = {\n    # Taiwan and related institutions consolidated to \'taiwan\'\n    \'taiwanese government\': \'taiwan\',\n    \'taiwan ministry of foreign affairs\': \'taiwan\',\n    "taiwan\'s defense ministry": \'taiwan\',\n    \'taiwan ministry of national defense\': \'taiwan\',\n    \'taiwan affairs office\': \'taiwan\',\n    \'taiwan government\': \'taiwan\',\n    \'taipei times\': \'taiwan\',\n    \'taiwan relations act\': \'us\',  # US legislation\n    \'taiwan ministry of national defence\': \'taiwan\',\n    "taiwan\'s defence ministry": \'taiwan\',\n    "taiwan\'s ministry of national defense": \'taiwan\',\n    \'taiwanese defense ministry\': \'taiwan\',\n    \'taiwan’s defense ministry\': \'taiwan\',\n    \'taiwan’s ministry of national defense\': \'taiwan\',\n\n    # China and variants consolidated to \'china\'\n    \'china\'s foreign ministry\': \'china\',\n    \'china’s foreign ministry\': \'china\',\n    \'chinese communist party\': \'china\',\n    \'chinese d

In [65]:
def consolidate_node(node, mapping):
    node_norm = str(node).lower().strip()
    return mapping.get(node_norm, node)

# Flatten edge_list of lists
flat_edge_list = [edge for sublist in edge_list for edge in sublist]

# Consolidate
consolidated_edges = [
    (consolidate_node(src, consolidation_dict), consolidate_node(tgt, consolidation_dict))
    for src, tgt in flat_edge_list
]


In [66]:
from collections import Counter
import netwulf

# Example: consolidated_edges = [...]  # list of (source, target) tuples

# Step 1: Count weighted edges
# If undirected, sort the pairs so (A,B) and (B,A) are treated the same
edge_counts = Counter(tuple(sorted((src, tgt))) for src, tgt in consolidated_edges if src and tgt)

# Step 2: Extract unique nodes from edge counts
unique_nodes = set()
for src, tgt in edge_counts:
    unique_nodes.update([src, tgt])

# Step 3: Create node dicts
nodes = [{"id": node} for node in unique_nodes]

# Step 4: Create weighted link dicts
links = [{"source": src, "target": tgt, "weight": weight} 
         for (src, tgt), weight in edge_counts.items()]

# Step 5: Build network and visualize
network = {"nodes": nodes, "links": links}
netwulf.visualize(network)

(None, None)

## Color Coding Clusters

## -based on entity

In [67]:
# dictionary for color coding
id2type = {
    # Political figures (people in politics, leaders)
    'alex azar': 'political_figure',
    'wang wenbin': 'political_figure',
    'antony blinken': 'political_figure',
    'audrey tang': 'political_figure',
    'bi-khim hsiao': 'political_figure',
    'bonnie glaser': 'think_tank',
    'david chou': 'political_figure',
    'donald trump': 'political_figure',
    'drew thompson': 'political_figure',
    'emmanuel macron': 'political_figure',
    'fumio kishida': 'political_figure',
    'gina raimondo': 'political_figure',
    'gordon chang': 'political_figure',
    'han kuo-yu': 'political_figure',
    'hua chunying': 'political_figure',
    'jake sullivan': 'political_figure',
    'jen psaki': 'political_figure',
    'joanne ou': 'political_figure',
    'joe biden': 'political_figure',
    'john cena': 'celebrity',  # actor/celebrity
    'john kirby': 'political_figure',
    'joseph wu': 'political_figure',
    'karine jean-pierre': 'political_figure',
    'kevin mccarthy': 'political_figure',
    'kurt campbell': 'political_figure',
    'lai ching-te': 'political_figure',
    'lev nachman': 'individual',  # unclear, assuming individual
    'lloyd austin': 'military',
    'mao ning': 'political_figure',
    'mark liu': 'political_figure',
    'mark milley': 'military',
    'marsha blackburn': 'political_figure',
    'michael mccaul': 'political_figure',
    'mike gallagher': 'political_figure',
    'mike pompeo': 'political_figure',
    'nancy pelosi': 'political_figure',
    'newt gingrich': 'political_figure',
    'nikki haley': 'political_figure',
    'ned price': 'political_figure',
    'terry gou': 'individual', # businessperson, treat as individual
    'tom cotton': 'political_figure',
    'vladimir putin': 'political_figure',
    'wang yi': 'political_figure',
    'zhao lijian': 'political_figure',
    'zhu fenglian': 'political_figure',
    'xi jinping': 'political_figure',
    'tsai ing-wen':'political_figure',

    # Celebrities, influencers (non-politicians)
    'john cena': 'celebrity',

    # Government agencies / departments / functions
    'taiwan': 'government_agency',
    'china': 'government_agency',
    'us': 'government_agency',
    'american institute in taiwan': 'government_agency',
    'biden administration': 'government_agency',
    'central epidemic command center': 'government_agency',
    'central news agency': 'government_agency',
    'chen shih-chung': 'government_agency',
    'congress': 'government_agency',
    'department of defense': 'government_agency',
    'democratic progressive party': 'political_party',
    'kuomintang': 'political_party',
    'labor department': 'government_agency',
    'nato': 'international_organization',
    'pentagon': 'government_agency',
    'state department': 'government_agency',
    'taipei times': 'media',
    'taiwan affairs office': 'government_agency',
    'taiwan creative content agency': 'government_agency',
    'taiwan defense ministry': 'government_agency',
    'taiwan government': 'government_agency',
    'taiwan ministry of national defense': 'government_agency',
    'taiwan stock exchange': 'government_agency',
    'united states government': 'government_agency',
    'us navy': 'military',
    'u.s. 7th fleet': 'military',
    'white house': 'government_agency',
    'u.s. government': 'government_agency',
    'u.s. military': 'military',
    'u.s. navy': 'military',
    'u.s. officials': 'government_agency',
    'center for strategic and international studies': 'think_tank',
    'central epidemic command center': 'government_agency',
    'commerce department': 'government_agency',
    'eastern theater command': 'military',
    'japan government': 'government_agency',
    'kuomintang': 'political_party',
    'new york times': 'media',
    'ntd': 'media',
    'reporters': 'media',
    'state department': 'government_agency',
    'taiwan affairs office': 'government_agency',
    'taiwan ministry of national defense': 'government_agency',
    'taiwan relations act': 'government_agency',
    'the guardian': 'media',
    'the new york times': 'media',
    'the wall street journal': 'media',
    'the washington post': 'media',
    'taiwanese defense ministry': 'government_agency',
    "taiwan's defence ministry": 'government_agency',
    "taiwan's defense ministry": 'government_agency',
    "taiwan's foreign ministry": 'government_agency',
    "taiwan's ministry of national defense": 'government_agency',
    'taiwanese government': 'government_agency',
    'beijing':'government_agency',
    'japan':"government_agency",
    

    # Military (organizations, commands, units)
    "people’s liberation army": 'military',
    "people's liberation army": 'military',
    "pla":"military",
    'chinese defense ministry': 'military',
    'chinese military': 'military',
    'people’s republic of china military': 'military',
    'eastern theater command': 'military',
    'u.s. indo-pacific command': 'military',
    'people’s liberation army': 'military',
    'taiwan’s defense ministry': 'military',
    'u.s. 7th fleet': 'military',
    'us navy': 'military',

    # International organizations
    'un': 'international_organization',
    'world health organization': 'international_organization',

    # Media / News agencies
    'ap': 'media',
    'global times':"media",
    'associated press': 'media',
    'bbc': 'media',
    'benzinga': 'media',
    'cbs news': 'media',
    'cctv': 'media',
    'cnn': 'media',
    'daily caller news foundation': 'media',
    'fox news': 'media',
    'fox news digital': 'media',
    'nbc news': 'media',
    'new york times': 'media',
    'nikkei asia': 'media',
    'npr': 'media',
    'politico': 'media',
    'reuters': 'media',
    'rtt news': 'media',
    'south china morning post': 'media',
    'taipei times': 'media',
    'the associated press': 'media',
    'the guardian': 'media',
    'the new york times': 'media',
    'the wall street journal': 'media',
    'the washington post': 'media',
    'variety': 'media',
    'washington post': 'media',
    'xinhua': 'media',
    'xinhua news agency':'media',
    'weibo': 'media',
    'wall street': 'media',
    'wall street journal': 'media',
    'bloomberg':"media",
    'russia':"government_agency",
    'australia':"government_agency",
    'japan':"government_agency",
    'financial times':"media",
    'federal reserve':"government_agency",

    # Private companies
    'apple': 'company',
    'foxconn': 'company',
    'intel': 'company',
    'taiwan semiconductor manufacturing co': 'company',
    'taiwan semiconductor manufacturing co.': 'company',
    'taiwan semiconductor manufacturing company': 'company',
    'united microelectronics corporation': 'company',
    'tsmc': 'company',

    # Unknown or empty
    'none': 'unknown',
    '': 'unknown',
    'who':"unknown",
}

type2col= {
    'political_figure': '#1f77b4',        # Blue
    'celebrity': '#ff7f0e',               # Orange
    'government_agency': '#2ca02c',       # Green
    'political_party': '#d62728',         # Red
    'military': '#9467bd',                 # Purple
    'international_organization': '#8c564b',  # Brown
    'media': '#e377c2',                   # Pink
    'company': '#7f7f7f',                 # Gray
    'individual': '#bcbd22',              # Olive
    'think_tank': '#17becf',              # Cyan
    'organization': '#aec7e8',            # Light Blue (general groups)
    'unknown': '#cccccc',                 # Light Gray for unknown or others
    'commodity': '#ffbb78',               # Light Orange for e.g. crude oil
    # Add more as needed
}

## -based on region

In [68]:
id2type = {
    # United States
    'alex azar': 'us',
    'american institute in taiwan': 'us',
    'antony blinken': 'us',
    'ap': 'us',
    'associated press': 'us',
    'biden administration': 'us',
    'bloomberg': 'us',
    'cbs news': 'us',
    'cnbc': 'us',
    'cnn': 'us',
    'commerce department': 'us',
    'congress': 'us',
    'daily caller news foundation': 'us',
    'department of defense': 'us',
    'donald trump': 'us',
    'cia': 'us',
    'elon musk': 'us',
    'federal reserve': 'us',
    'fox news': 'us',
    'fox news digital': 'us',
    'gina raimondo': 'us',
    'gordon chang': 'us',
    'jake sullivan': 'us',
    'jen psaki': 'us',
    'joe biden': 'us',
    'john kirby': 'us',
    'karine jean-pierre': 'us',
    'kevin mccarthy': 'us',
    'kurt campbell': 'us',
    'labor department': 'us',
    'lloyd austin': 'us',
    'mark milley': 'us',
    'marsha blackburn': 'us',
    'michael mccaul': 'us',
    'mike gallagher': 'us',
    'mike pompeo': 'us',
    'nancy pelosi': 'us',
    'nbc news': 'us',
    'ned price': 'us',
    'new york times': 'us',
    'newt gingrich': 'us',
    'npr': 'us',
    'pentagon': 'us',
    'politico': 'us',
    'state department': 'us',
    'the associated press': 'us',
    'the new york times': 'us',
    'wall street journal': 'us',
    'the washington post': 'us',
    'tom cotton': 'us',
    'trump administration': 'us',
    'u.s.': 'us',
    'u.s. 7th fleet': 'us',
    'u.s. government': 'us',
    'u.s. indo-pacific command': 'us',
    'u.s. military': 'us',
    'u.s. navy': 'us',
    'u.s. officials': 'us',
    'u.s. state department': 'us',
    'united states': 'us',
    'us navy': 'us',
    'white house': 'us',
    'wall street': 'us',
    'us':'us',
    'rtt news':'us',
    'rtt':'us',
    'benzinga':'us',
    'taiwan relations act': 'us',

    # China / PRC
    'beijing': 'china',
    'cctv': 'china',
    'china\'s foreign ministry': 'china',
    'china’s foreign ministry': 'china',
    'chinese communist party': 'china',
    'chinese defense ministry': 'china',
    'chinese embassy': 'china',
    'chinese foreign ministry': 'china',
    'chinese government': 'china',
    'chinese military': 'china',
    'chinese state media': 'china',
    'eastern theater command': 'china',
    'global times': 'china',
    'hua chunying': 'china',
    'mao ning': 'china',
    'people\'s liberation army': 'china',
    'people\'s republic of china': 'china',
    'people’s liberation army': 'china',
    'people’s republic of china': 'china',
    'qin gang': 'china',
    'wang wenbin': 'china',
    'wang yi': 'china',
    'wei fenghe': 'china',
    'weibo': 'china',
    'xinhua': 'china',
    'xinhua news agency': 'china',
    'xi jinping': 'china',
    'zhao lijian': 'china',
    'zhu fenglian': 'china',
    'taiwan affairs office': 'china',
    'pla':'china',
    'south china morning post':'china',
    'china':'china',
    

    # Taiwan / ROC
    'audrey tang': 'taiwan',
    'bi-khim hsiao': 'taiwan',
    'center for strategic and international studies': 'us',  # US-based but policy-focused on Taiwan
    'central epidemic command center': 'taiwan',
    'central news agency': 'taiwan',
    'chen shih-chung': 'taiwan',
    'chiu kuo-cheng': 'taiwan',
    'democratic progressive party': 'taiwan',
    'david chou': 'taiwan',
    'han kuo-yu': 'taiwan',
    'joanne ou': 'taiwan',
    'joseph wu': 'taiwan',
    'kuomintang': 'taiwan',
    'lai ching-te': 'taiwan',
    'lev nachman': 'taiwan',  # Political scientist based in Taiwan
    'mark liu': 'taiwan',
    'su tseng-chang': 'taiwan',
    'taipei times': 'taiwan',
    'taiwan': 'taiwan',
    'taiwan creative content agency': 'taiwan',
    'taiwan defense ministry': 'taiwan',
    'taiwan government': 'taiwan',
    'taiwan ministry of national defense': 'taiwan',
    'taiwan news': 'taiwan',
    'taiwan presidential office': 'taiwan',
    
    'taiwan semiconductor manufacturing co': 'taiwan',
    'taiwan semiconductor manufacturing co.': 'taiwan',
    'taiwan semiconductor manufacturing company': 'taiwan',
    'taiwan stock exchange': 'taiwan',
    'taiwan\'s defence ministry': 'taiwan',
    'taiwan\'s defense ministry': 'taiwan',
    'taiwan\'s foreign ministry': 'taiwan',
    'taiwan\'s government': 'taiwan',
    'taiwan\'s ministry of national defense': 'taiwan',
    'taiwanese defense ministry': 'taiwan',
    'taiwanese government': 'taiwan',
    'taiwan’s defense ministry': 'taiwan',
    'taiwan’s ministry of national defense': 'taiwan',
    'terry gou': 'taiwan',
    'tsai ing-wen': 'taiwan',
    'tsmc': 'taiwan',
    'william lai': 'taiwan',

    # Japan
    'fumio kishida': 'japan',
    'japan': 'japan',
    'nikkei asia': 'japan',

    # Australia
    'australia': 'australia',

    # Russia
    'vladimir putin': 'russia',
    'russia': 'russia',

    # Philippines
    'philippines': 'philippines',

    # Europe 
    'emmanuel macron': 'europe',
    

    # UK 
    'bbc': 'uk',
    'the guardian': 'uk',
    'reuters':"uk",
    'financial times':'uk',
    

    # Multinational / Global
    'apple': 'multinational',
    'intel': 'multinational',
    'united microelectronics corporation': 'multinational',
    'variety': 'multinational',
    'world health organization': 'multinational',
    'united nations': 'multinational',

    # Ambiguous or Unknown
    'none': 'unknown',
    'rtt news': 'unknown',
    'bonnie glaser': 'unknown',  # Could be US-based but focused on East Asia
    'john cena': 'us',  # US citizen, global actor
    'drew thompson': 'us',
    'financial times': 'uk',
    'south china morning post': 'hong_kong',
    'reuters': 'uk',
    'washington post': 'us',
    'west texas intermediate crude': 'us'
}

type2col = {
    'us': '#1f77b4',                      # Blue - United States
    'china': '#d62728',                   # Red - China
    'taiwan': '#2ca02c',                  # Green - Taiwan
    'europe': '#9467bd',                  # Purple - Europe
    'uk' : '#ffb6c1',                     # light Pink - UK
    'japan': '#ff7f0e',                   # Orange - Japan
    'australia': '#8c564b',               # Brown - Australia
    'israel': '#17becf',                  # Cyan - Israel
    'palestine': '#e377c2',               # Pink - Palestine
    'russia': '#7f7f7f',                  # Gray - Russia
    'philippines': '#bcbd22',             # Olive - Philippines
    'qatar': '#aec7e8',                   # Light Blue - Qatar
    'lebanon': '#f7b6d2',                 # Light Pink - Lebanon
    'multinational': '#9edae5',           # Teal - UN or int’l orgs
    'media_us': '#c5b0d5',                # Light Purple - US media
    'media_china': '#ff9896',             # Light Red - China media
    'media_taiwan': '#98df8a',            # Light Green - Taiwan media
    'media_other': '#c7c7c7',             # Light Gray - Global/neutral media
    'corporation': '#fdbf6f',             # Gold - Corporations
    'political_figure': '#393b79',        # Dark Blue - Political actors
    'think_tank': '#6b6ecf',              # Blue-violet - Policy/Research
    'military': '#525252',                # Dark Gray - Military
    'government': '#1f78b4',              # Blue - Default for gov actors
    'social_media_user': '#ffbb78',       # Light Orange - Citizens/influencers
    'academic': '#17becf',                # Cyan - Scholars
    'unknown': '#d9d9d9',                 # Light Gray - Unknown/misc
}

In [69]:
"""
import networkx as nx

G = nx.Graph()
for (src, tgt), weight in edge_counts.items():
    G.add_edge(src, tgt, weight=weight)

# Step 3: Add node attributes: type and color
for node in G.nodes():
    node_type = id2type.get(node.lower(), "Unknown")  # lowercase keys
    G.nodes[node]["type"] = node_type
    G.nodes[node]["color"] = type2col.get(node_type, "gray")

# Step 4: Convert NetworkX graph to Netwulf data format
def networkx_to_netwulf_data(G):
    nodes = []
    for n, attr in G.nodes(data=True):
        nodes.append({
            "id": n,
            "label": n,
            "color": attr.get("color", "gray"),
            "type": attr.get("type", "Unknown"),
            "size": 10  # or set size based on some attribute
        })
    
    links = []
    for s, t, attr in G.edges(data=True):
        links.append({
            "source": s,
            "target": t,
            "weight": attr.get("weight", 1)
        })
    
    return {"nodes": nodes, "links": links}

network = networkx_to_netwulf_data(G_U1)

# Step 5: Visualize with Netwulf
netwulf.visualize(network)
"""

'\nimport networkx as nx\n\nG = nx.Graph()\nfor (src, tgt), weight in edge_counts.items():\n    G.add_edge(src, tgt, weight=weight)\n\n# Step 3: Add node attributes: type and color\nfor node in G.nodes():\n    node_type = id2type.get(node.lower(), "Unknown")  # lowercase keys\n    G.nodes[node]["type"] = node_type\n    G.nodes[node]["color"] = type2col.get(node_type, "gray")\n\n# Step 4: Convert NetworkX graph to Netwulf data format\ndef networkx_to_netwulf_data(G):\n    nodes = []\n    for n, attr in G.nodes(data=True):\n        nodes.append({\n            "id": n,\n            "label": n,\n            "color": attr.get("color", "gray"),\n            "type": attr.get("type", "Unknown"),\n            "size": 10  # or set size based on some attribute\n        })\n    \n    links = []\n    for s, t, attr in G.edges(data=True):\n        links.append({\n            "source": s,\n            "target": t,\n            "weight": attr.get("weight", 1)\n        })\n    \n    return {"nodes": no

In [70]:
import netwulf
import networkx as nx

G = nx.Graph()
for (src, tgt), weight in edge_counts.items():
    G.add_edge(src, tgt, weight=weight)

# 1. Assign colors to nodes
for k, v in G.nodes(data=True):
    node_type = id2type.get(k.lower(), 'unknown')         # use .lower() if keys are lowercase
    color = type2col.get(node_type, '#cccccc')            # fallback gray color
    v["color"] = color                                    # store in "color" attribute

# 2. Convert NetworkX graph to Netwulf data format (nodes + links with colors)
def networkx_to_netwulf_data(G):
    nodes = []
    for n, attr in G.nodes(data=True):
        nodes.append({
            "id": n,
            "label": n,
            "color": attr.get("color", "#cccccc"),
            "size": 10
        })
    links = []
    for s, t, attr in G.edges(data=True):
        links.append({
            "source": s,
            "target": t,
            "weight": attr.get("weight", 1)
        })
    return {"nodes": nodes, "links": links}

# 3. Create Netwulf data dict
stylized_network = networkx_to_netwulf_data(G)

config = {
    'node_label_font_size': 10,  # default is 10, increase for better visibility
}

# 4. Visualize (Netwulf returns the stylized network data and config)
stylized_network, config = netwulf.visualize(stylized_network,config=config)

## Original Batching

In [72]:
import json
import pandas as pd

# Step 1: Load all results from multiple files
results = []
file_paths = [
    "Batching_Jsonl_Files/Final/Output_A.jsonl",
    "Batching_Jsonl_Files/Final/Output_B.jsonl",
    "Batching_Jsonl_Files/Final/Output_C.jsonl"
]

for path in file_paths:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            results.append(data)

print(f"✅ Loaded {len(results)} results from {len(file_paths)} files.")

# Step 2: Filter and parse valid responses (not skipped)
valid_data = []
skipped_count = 0
invalid_format_count = 0

for res in results:
    response = res.get("response", {})
    content = response.get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
    try:
        answer_json = json.loads(content)
        if answer_json.get("skip") is not True:
            answer_json["id"] = res.get("custom_id")
            valid_data.append(answer_json)
        else:
            skipped_count += 1
    except json.JSONDecodeError:
        invalid_format_count += 1
        print(f"⚠️ JSON decode error for ID {res.get('custom_id')}")

# Step 3: Create DataFrame
dfbr = pd.DataFrame(valid_data)

# Step 4: Print summary
print(f"✅ Valid (not skipped): {len(valid_data)}")
print(f"🚫 Skipped: {skipped_count}")
print(f"❌ Invalid JSON: {invalid_format_count}")

# Step 5: (Optional) preview results
dfbr.head()


In [73]:
dfbr.head(20)

In [74]:
dfbr.columns

In [75]:
dfbr.shape

In [76]:
merged_df = pd.merge(df, dfbr, left_on="sid", right_on="id", how="inner")
merged_df.head(10)

In [77]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Identify the string columns to exclude
string_cols = [
    'author', 'top_actor_1', 'top_actor_2', 'top_actor_3',
    'top_source_1', 'top_source_2', 'top_source_3', 'id'
]

# Step 2: Select only numeric columns
numeric_df = dfbr.drop(columns=string_cols, errors='ignore')

# Step 3: Compute the correlation matrix
correlation_matrix = numeric_df.corr()

# Step 4: Plot the heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.5}
)
plt.title("Correlation Heatmap of Taiwan Media Coverage Variables", fontsize=16)
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [78]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Drop non-numeric columns
numeric_df = dfbr.select_dtypes(include=[np.number])

# Step 1: Compute correlation matrix and p-values
corr = numeric_df.corr()
pvals = pd.DataFrame(np.ones(corr.shape), columns=corr.columns, index=corr.index)

for row in corr.columns:
    for col in corr.columns:
        if row != col:
            _, p = pearsonr(numeric_df[row], numeric_df[col])
            pvals.loc[row, col] = p
        else:
            pvals.loc[row, col] = np.nan  # Set diagonal to NaN for clarity

# Step 2: Define function to turn p-values into significance stars
def significance_stars(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    else:
        return ''

# Step 3: Create annotations (e.g., 0.67**)
annot = corr.round(2).astype(str)
for i in corr.columns:
    for j in corr.columns:
        if i != j:
            stars = significance_stars(pvals.loc[i, j])
            annot.loc[i, j] += stars
        else:
            annot.loc[i, j] = ''  # Leave diagonal blank

# Step 4: Plot heatmap with stars
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=annot, fmt='', cmap='coolwarm', center=0, square=True,
            cbar_kws={"shrink": .8}, linewidths=.5, annot_kws={"fontsize": 6})
plt.title('Correlation Matrix with Significance Stars', fontsize=16)
plt.tight_layout()
plt.show()

## Authors ##

In [79]:
dfbr["author"].value_counts(dropna=True).head(10)

In [80]:
dfbrr["author"].value_counts(dropna=True).head(10)

author
AP                  86
Associated Press    79
Frances Martel      79
Ben Blanchard       48
Ian Hanchett        38
CNN                 34
Kristina Wong       29
Joe Gould           27
Huizhong Wu         25
Josh Rogin          23
Name: count, dtype: int64

## Most cited actors 

In [81]:
import numpy as np

In [82]:
dfbrr["top_actor_1"].value_counts(dropna=True).head(20)

top_actor_1
China                                     2257
Joe Biden                                 1179
Nancy Pelosi                               923
Taiwan                                     690
Xi Jinping                                 531
United States                              500
Tsai Ing-wen                               475
Taiwan Stock Exchange                      341
Antony Blinken                             196
Biden administration                       151
Donald Trump                               134
TSMC                                       132
Kevin McCarthy                             100
Lloyd Austin                                96
Chinese Communist Party                     95
World Health Organization                   84
Taiwan Semiconductor Manufacturing Co.      73
U.S. Navy                                   71
Mike Pompeo                                 60
Alex Azar                                   59
Name: count, dtype: int64

In [83]:
dfbrr["top_actor_2"].value_counts(dropna=True).head(20)

top_actor_2
China                                         2554
Taiwan                                        1379
United States                                  960
Xi Jinping                                     958
Joe Biden                                      488
Tsai Ing-wen                                   326
Nancy Pelosi                                   269
Chinese Communist Party                        255
Taiwan Semiconductor Manufacturing Company     163
Donald Trump                                   108
World Health Organization                      100
Federal Reserve                                 97
Democratic Progressive Party                    94
Kevin McCarthy                                  87
Russia                                          87
Biden administration                            84
Beijing                                         80
Chinese government                              69
White House                                     67
Antony Blinken     

In [84]:
dfbrr["top_actor_3"].value_counts(dropna=True).head(20)

top_actor_3
Taiwan                                        1750
China                                         1580
United States                                 1089
Xi Jinping                                     599
Joe Biden                                      434
Tsai Ing-wen                                   401
Nancy Pelosi                                   352
Chinese Communist Party                        265
Russia                                         155
Beijing                                        143
Biden administration                           117
People's Liberation Army                       115
Donald Trump                                   112
People’s Liberation Army                        92
U.S.                                            92
White House                                     86
Democratic Progressive Party                    73
                                                72
World Health Organization                       71
Taiwan Semiconducto

## Cleaning nasty actors and sources

In [85]:
from remapper1 import *

In [86]:
dfbrr["top_actor_1"].unique()
#dfbr["top1_cleaned"] = dfbrr.top_actor_1.apply(lambda x: flat_mapping.get(x, x) )

array(['China', 'David Wenwei Chou', 'Nancy Pelosi', ...,
       'Michael Minihan', 'Bulgari', 'Henry A. Kissinger'], dtype=object)

In [87]:
dfbr["top_actor_1"].unique()

In [88]:
dfbr["top1_cleaned"] = dfbr.top_actor_1.apply(lambda x: flat_mapping.get(x, x) )

In [89]:
v = np.hstack((dfbr.top_actor_1.astype(str).values, dfbr.top_actor_2.astype(str).values, dfbr.top_actor_3.astype(str).values))
v = [vv.split(",")[0] for vv in v]
v = np.unique(v)
print(len(v))
print(";\n".join(v) )

In [90]:
print( ";\n".join( dfbr.top_actor_1.value_counts().iloc[:400].index  ) )

In [91]:
dfbrr

,author,top_actor_1,top_actor_2,top_actor_3,top_source_1,top_source_2,top_source_3,id
0,None,China,Taiwan,People’s Liberation Army,Taipei-based China Times,unnamed military officials,,c71b80d7506d468f75f501a08e5fc1b853903aba9fa6cb...
1,Emily Feng,David Wenwei Chou,Tony Lee,Jenny Koo,Tony Lee,Jenny Koo,Leona Chen,5b7d709fd7773ac57486c89818415d09bfbdb8b011fcc8...
2,Kristina Wong,Nancy Pelosi,Biden administration,China,Nancy Pelosi,Hu Xijin,Republicans,41525d643766c220e21cdef20d2fdbfe57d8f9d0913109...
3,Rich Edson,Taiwan,World Health Organization,China,Taiwan's Center for Disease Control and Preven...,World Health Organization,Chinese CDC,39cad927166a649f47f91051c8664416e8171703dcbd92...
4,None,Yoshimasa Hayashi,China,Russia,Centre for Strategic and International Studies,Japan-US alliance,Tokyo government,96fe9f697b8df4fdab27aab37b63d4451c179fc0295c0d...
...,...,...,...,...,...,...,...,...
12811,Associated Press,Joe Biden,Mao Ning,Xi Jinping,CBS News,Taiwan’s Foreign Ministry,U.S. Foreign Ministry,61d744e3d7805717bedff1ce75f0513b4d278c4df32751...
12812,None,China,United States,Taiwan,Pentagon,Gen. Mark Milley,Federation of American Scientists,05fe8fdbed0b48936712ed11348c2f6183fe5ae325723a...
12813,Huizhong Wu,Chen Chi Mai,Tsai Hsiu-Chin,Lin Chie-ying,Associated Press,Taiwan’s Central News Agency,United Daily News,ed885a5cddda1ccd8bb9c5ebe71a551502cc5489e4c62e...
12814,None,China,Taiwan,United States,Mao Ning,Tsai Ing-wen,Kevin McCarthy,13c0ab35e61ec25c17810db22e6f5aa49dffab678722e9...


In [92]:
dfbrr["top_actor_2"].value_counts(dropna=True).head(10)

top_actor_2
China                                         2554
Taiwan                                        1379
United States                                  960
Xi Jinping                                     958
Joe Biden                                      488
Tsai Ing-wen                                   326
Nancy Pelosi                                   269
Chinese Communist Party                        255
Taiwan Semiconductor Manufacturing Company     163
Donald Trump                                   108
Name: count, dtype: int64

In [93]:
dfbrr["top_actor_3"].value_counts(dropna=True).head(10)

top_actor_3
Taiwan                     1750
China                      1580
United States              1089
Xi Jinping                  599
Joe Biden                   434
Tsai Ing-wen                401
Nancy Pelosi                352
Chinese Communist Party     265
Russia                      155
Beijing                     143
Name: count, dtype: int64

## Most cited sources 

In [94]:
dfbrr["top_source_1"].value_counts(dropna=True).head(10)

top_source_1
Reuters           320
White House       296
CNN               260
Global Times      194
Joe Biden         163
Fox News          147
RTT News          138
Pentagon          136
Nancy Pelosi      133
Antony Blinken    117
Name: count, dtype: int64

In [95]:
dfbrr["top_source_2"].value_counts(dropna=True).head(10)

top_source_2
Chinese Foreign Ministry    217
Tsai Ing-wen                197
Reuters                     176
White House                 166
Joe Biden                   157
Global Times                120
Xi Jinping                  120
John Kirby                  113
Antony Blinken              110
Fox News                    104
Name: count, dtype: int64

In [96]:
dfbrr["top_source_3"].value_counts(dropna=True).head(10)

top_source_3
                            211
Reuters                     174
Xi Jinping                  142
White House                 130
Tsai Ing-wen                124
China                       123
Fox News                    109
Joe Biden                   105
Chinese Foreign Ministry    104
Antony Blinken              103
Name: count, dtype: int64

# Centrality

In [97]:
B = pd.DataFrame( nx.betweenness_centrality(G).items(), columns =["source","betweenness"] )
D = pd.DataFrame( nx.degree_centrality(G).items(), columns =["source","degree"] )
B = B.merge(D)

In [98]:
G.edges()

EdgeView([('new york times', 'the associated press'), ('new york times', 'bloomberg'), ('new york times', 'wall street journal'), ('new york times', 'bbc'), ('new york times', 'mike pompeo'), ('new york times', 'wang wenbin'), ('new york times', 'white house'), ('new york times', 'tsmc'), ('new york times', 'weibo'), ('new york times', 'washington post'), ('new york times', 'cnn'), ('new york times', 'china'), ('new york times', 'cnbc'), ('new york times', 'ap'), ('new york times', 'state department'), ('new york times', 'fox news'), ('new york times', 'joseph wu'), ('new york times', 'chinese foreign ministry'), ('new york times', 'none'), ('new york times', 'global times'), ('new york times', 'cia'), ('new york times', 'center for strategic and international studies'), ('new york times', 'who'), ('new york times', 'financial times'), ('new york times', 'department of defense'), ('new york times', 'nbc news'), ('new york times', 'john kirby'), ('new york times', 'xi jinping'), ('new y

In [99]:
B.sort_values("betweenness", ascending=False)[0:10]

,source,betweenness,degree
22,cnn,0.046577,0.662651
3,china,0.037569,0.771084
17,chinese foreign ministry,0.034924,0.783133
36,xi jinping,0.032538,0.662651
14,white house,0.028317,0.710843
47,taiwan's defense ministry,0.026506,0.722892
48,pentagon,0.025870,0.602410
8,fox news,0.021312,0.650602
11,joe biden,0.021025,0.614458
13,wall street journal,0.020421,0.566265


## Country Level Meso Network

## Source 

In [149]:
source_country_mapping = {
    "american institute in taiwan": "United States",
    "antony blinken": "United States",
    "ap": "United States",
    "associated press": "United States",
    "bbc": "United Kingdom",
    "beijing": "China",
    "benzinga": "United States",
    "bi-khim hsiao": "Taiwan",
    "biden administration": "United States",
    "bloomberg": "United States",
    "bonnie glaser": "United States",
    "cbs news": "United States",
    "cctv": "China",
    "center for strategic and international studies": "United States",
    "central news agency": "Taiwan",
    "china": "China",
    "china's foreign ministry": "China",
    "china’s foreign ministry": "China",
    "chinese communist party": "China",
    "chinese defense ministry": "China",
    "chinese embassy": "China",
    "chinese foreign ministry": "China",
    "chinese government": "China",
    "chinese state media": "China",
    "chiu kuo-cheng": "Taiwan",
    "cia": "United States",
    "cnbc": "United States",
    "cnn": "United States",
    "commerce department": "United States",
    "daily caller news foundation": "United States",
    "department of defense": "United States",
    "donald trump": "United States",
    "drew thompson": "United States",
    "financial times": "United Kingdom",
    "fox news": "United States",
    "fox news digital": "United States",
    "global times": "China",
    "gordon chang": "United States",
    "hua chunying": "China",
    "jake sullivan": "United States",
    "jen psaki": "United States",
    "joanne ou": "Taiwan",
    "joe biden": "United States",
    "john kirby": "United States",
    "joseph wu": "Taiwan",
    "karine jean-pierre": "United States",
    "kevin mccarthy": "United States",
    "labor department": "United States",
    "lev nachman": "United States",
    "lloyd austin": "United States",
    "mao ning": "China",
    "mike pompeo": "United States",
    "nancy pelosi": "United States",
    "nbc news": "United States",
    "new york times": "United States",
    "none": None,
    "npr": "United States",
    "pentagon": "United States",
    "people’s liberation army": "China",
    "politico": "United States",
    "qin gang": "China",
    "rtt news": "United States",
    "south china morning post": "Hong Kong",
    "state department": "United States",
    "taipei times": "Taiwan",
    "taiwan": "Taiwan",
    "taiwan affairs office": "China",
    "taiwan defense ministry": "Taiwan",
    "taiwan news": "Taiwan",
    "taiwan presidential office": "Taiwan",
    "taiwan relations act": "United States",
    "taiwan's defence ministry": "Taiwan",
    "taiwan's defense ministry": "Taiwan",
    "taiwan's foreign ministry": "Taiwan",
    "taiwan's ministry of national defense": "Taiwan",
    "taiwanese defense ministry": "Taiwan",
    "taiwan’s defense ministry": "Taiwan",
    "taiwan’s ministry of national defense": "Taiwan",
    "the associated press": "United States",
    "the guardian": "United Kingdom",
    "the new york times": "United States",
    "the wall street journal": "United States",
    "the washington post": "United States",
    "tsai ing-wen": "Taiwan",
    "tsmc": "Taiwan",
    "u.s. 7th fleet": "United States",
    "u.s. navy": "United States",
    "u.s. officials": "United States",
    "u.s. state department": "United States",
    "united states": "United States",
    "variety": "United States",
    "wall street": "United States",
    "wall street journal": "United States",
    "wang wenbin": "China",
    "wang yi": "China",
    "washington post": "United States",
    "weibo": "China",
    "west texas intermediate crude": "United States",
    "white house": "United States",
    "world health organization": "International",
    "xi jinping": "China",
    "xinhua": "China",
    "xinhua news agency": "China",
    "zhao lijian": "China",
    "zhu fenglian": "China"
}

In [150]:
def consolidate_node(node, mapping):
    node_norm = str(node).lower().strip()
    return mapping.get(node_norm, node)

# Flatten edge_list of lists
flat_edge_list = [edge for sublist in edge_list for edge in sublist]

# Consolidate
consolidated_edges = [
    (consolidate_node(src, source_country_mapping), consolidate_node(tgt, source_country_mapping))
    for src, tgt in flat_edge_list
]


In [151]:
from collections import Counter
import netwulf

# Example: consolidated_edges = [...]  # list of (source, target) tuples

# Step 1: Count weighted edges
# If undirected, sort the pairs so (A,B) and (B,A) are treated the same
edge_counts = Counter(tuple(sorted((src, tgt))) for src, tgt in consolidated_edges if src and tgt)

# Step 2: Extract unique nodes from edge counts
unique_nodes = set()
for src, tgt in edge_counts:
    unique_nodes.update([src, tgt])

# Step 3: Create node dicts
nodes = [{"id": node} for node in unique_nodes]

# Step 4: Create weighted link dicts
links = [{"source": src, "target": tgt, "weight": weight} 
         for (src, tgt), weight in edge_counts.items()]

# Step 5: Build network and visualize
network = {"nodes": nodes, "links": links}
netwulf.visualize(network)

(None, None)

## Actor

In [136]:
actor_country_mapping = {
    "alex azar": "United States",
    "antony blinken": "United States",
    "apple": "United States",
    "australia": "Australia",
    "beijing": "China",
    "biden administration": "United States",
    "chen shih-chung": "Taiwan",
    "chinese communist party": "China",
    "chinese government": "China",
    "chinese military": "China",
    "david chou": "Taiwan",
    "democratic progressive party": "Taiwan",
    "donald trump": "United States",
    "federal reserve": "United States",
    "foxconn": "Taiwan",
    "fumio kishida": "Japan",
    "han kuo-yu": "Taiwan",
    "jake sullivan": "United States",
    "japan": "Japan",
    "joe biden": "United States",
    "john cena": "United States",
    "joseph wu": "Taiwan",
    "kevin mccarthy": "United States",
    "kuomintang": "Taiwan",
    "lloyd austin": "United States",
    "michael mccaul": "United States",
    "mike gallagher": "United States",
    "mike pompeo": "United States",
    "nancy pelosi": "United States",
    "pentagon": "United States",
    "people's liberation army": "China",
    "people's republic of china": "China",
    "people’s liberation army": "China",
    "philippines": "Philippines",
    "qin gang": "China",
    "russia": "Russia",
    "taiwan": "Taiwan",
    "taiwan government": "Taiwan",
    "taiwan semiconductor manufacturing co.": "Taiwan",
    "taiwan semiconductor manufacturing company": "Taiwan",
    "taiwan stock exchange": "Taiwan",
    "taiwan's defense ministry": "Taiwan",
    "taiwanese government": "Taiwan",
    "trump administration": "United States",
    "tsai ing-wen": "Taiwan",
    "tsmc": "Taiwan",
    "u.s.": "United States",
    "u.s. military": "United States",
    "u.s. navy": "United States",
    "united microelectronics corporation": "Taiwan",
    "united states": "United States",
    "vladimir putin": "Russia",
    "wall street": "United States",
    "wang yi": "China",
    "white house": "United States",
    "william lai": "Taiwan",
    "world health organization": "International",
    "xi jinping": "China"
}

In [137]:
import numpy as np
from itertools import combinations

def consolidate_node(node, mapping):
    node_norm = str(node).lower().strip()
    return mapping.get(node_norm, node)

# Actor co-mention edges. (This cell previously reused the *source* edge list, so it never built an actor network.)
actor_cols = ["top_actor_1", "top_actor_2", "top_actor_3"]
_vals = pd.Series([str(x).lower() for x in np.hstack(dfbrr[actor_cols].values)])
_counts = _vals.value_counts()
actor_sub = set(_counts[_counts > 50].index[1:])
actor_edge_list = []
for row in dfbrr[actor_cols].itertuples(index=False):
    nodes = [str(d).lower() for d in row if str(d).lower() in actor_sub]
    actor_edge_list.append(list(combinations(nodes, 2)))
flat_edge_list = [edge for sublist in actor_edge_list for edge in sublist]

if not actor_country_mapping:
    print("WARNING: actor_country_mapping is empty - fill it in (it may live in remapper1.py) or nodes will not be grouped by country.")

consolidated_edges = [
    (consolidate_node(src, actor_country_mapping), consolidate_node(tgt, actor_country_mapping))
    for src, tgt in flat_edge_list
]


In [138]:
from collections import Counter
import netwulf

# Example: consolidated_edges = [...]  # list of (source, target) tuples

# Step 1: Count weighted edges
# If undirected, sort the pairs so (A,B) and (B,A) are treated the same
edge_counts = Counter(tuple(sorted((src, tgt))) for src, tgt in consolidated_edges if src and tgt)

# Step 2: Extract unique nodes from edge counts
unique_nodes = set()
for src, tgt in edge_counts:
    unique_nodes.update([src, tgt])

# Step 3: Create node dicts
nodes = [{"id": node} for node in unique_nodes]

# Step 4: Create weighted link dicts
links = [{"source": src, "target": tgt, "weight": weight} 
         for (src, tgt), weight in edge_counts.items()]

# Step 5: Build network and visualize
network = {"nodes": nodes, "links": links}
netwulf.visualize(network)

(None, None)